In [0]:
%sql
CREATE OR REPLACE TABLE medical_insurance.gold.doctor_performance AS
WITH doctor_base AS (
  SELECT 
    d.doctor_id,
    d.doctor_name,
    d.hospital_id,
    d.department_id,
    d.specialty,
    d.years_experience,
    d.employment_status
  FROM medical_insurance.silver.doctor_silver d
),
visit_stats AS (
  SELECT 
    v.doctor_id,
    COUNT(DISTINCT v.visit_id) AS total_visits,
    COUNT(DISTINCT CASE WHEN v.visit_type = 'Outpatient' THEN v.visit_id END) AS outpatient_visits,
    COUNT(DISTINCT CASE WHEN v.visit_type = 'Inpatient' THEN v.visit_id END) AS inpatient_visits,
    COUNT(DISTINCT CASE WHEN v.visit_type = 'Follow-up' THEN v.visit_id END) AS followup_visits,
    COUNT(DISTINCT v.patient_id) AS unique_patients,
    AVG(v.waiting_time) AS avg_waiting_time_minutes,
    SUM(v.total_amount) AS total_revenue,
    AVG(v.total_amount) AS avg_revenue_per_visit,
    MIN(v.visit_date) AS first_visit_date,
    MAX(v.visit_date) AS last_visit_date,
    COUNT(DISTINCT v.diagnosis_code) AS unique_diagnoses_handled
  FROM medical_insurance.silver.visit_silver v
  GROUP BY v.doctor_id
),
feedback_stats AS (
  SELECT 
    f.doctor_id,
    COUNT(DISTINCT f.feedback_id) AS feedback_count,
    AVG(f.rating) AS avg_patient_rating,
    MIN(f.rating) AS min_rating,
    MAX(f.rating) AS max_rating,
    COUNT(DISTINCT CASE WHEN f.rating >= 4 THEN f.feedback_id END) AS positive_feedback_count,
    COUNT(DISTINCT CASE WHEN f.rating <= 2 THEN f.feedback_id END) AS negative_feedback_count
  FROM medical_insurance.silver.patient_feedback_silver f
  GROUP BY f.doctor_id
),
schedule_stats AS (
  SELECT 
    s.doctor_id,
    COUNT(DISTINCT s.schedule_id) AS total_shifts,
    COUNT(DISTINCT s.shift_date) AS days_scheduled,
    MIN(s.shift_date) AS first_shift_date,
    MAX(s.shift_date) AS last_shift_date
  FROM medical_insurance.silver.doctor_schedule_silver s
  GROUP BY s.doctor_id
),
hospital_dept_info AS (
  SELECT 
    d.doctor_id,
    h.hospital_name,
    h.hospital_type,
    dept.department_name
  FROM medical_insurance.silver.doctor_silver d
  LEFT JOIN medical_insurance.silver.hospital_silver h ON d.hospital_id = h.hospital_id
  LEFT JOIN medical_insurance.silver.department_silver dept ON d.department_id = dept.department_id
)
SELECT 
  db.doctor_id,
  db.doctor_name,
  db.hospital_id,
  hd.hospital_name,
  hd.hospital_type,
  db.department_id,
  hd.department_name,
  db.specialty,
  db.years_experience,
  db.employment_status,
  
  -- Visit metrics
  COALESCE(vs.total_visits, 0) AS total_visits,
  COALESCE(vs.outpatient_visits, 0) AS outpatient_visits,
  COALESCE(vs.inpatient_visits, 0) AS inpatient_visits,
  COALESCE(vs.followup_visits, 0) AS followup_visits,
  COALESCE(vs.unique_patients, 0) AS unique_patients_treated,
  ROUND(COALESCE(vs.avg_waiting_time_minutes, 0), 2) AS avg_waiting_time_minutes,
  vs.first_visit_date,
  vs.last_visit_date,
  COALESCE(vs.unique_diagnoses_handled, 0) AS unique_diagnoses_handled,
  
  -- Revenue metrics
  ROUND(COALESCE(vs.total_revenue, 0), 2) AS total_revenue,
  ROUND(COALESCE(vs.avg_revenue_per_visit, 0), 2) AS avg_revenue_per_visit,
  
  -- Patient satisfaction metrics
  COALESCE(fs.feedback_count, 0) AS feedback_count,
  ROUND(COALESCE(fs.avg_patient_rating, 0), 2) AS avg_patient_rating,
  fs.min_rating,
  fs.max_rating,
  COALESCE(fs.positive_feedback_count, 0) AS positive_feedback_count,
  COALESCE(fs.negative_feedback_count, 0) AS negative_feedback_count,
  CASE 
    WHEN fs.feedback_count > 0 THEN ROUND((fs.positive_feedback_count * 100.0 / fs.feedback_count), 2)
    ELSE 0
  END AS positive_feedback_rate_pct,
  
  -- Schedule metrics
  COALESCE(ss.total_shifts, 0) AS total_shifts,
  COALESCE(ss.days_scheduled, 0) AS days_scheduled,
  ss.first_shift_date,
  ss.last_shift_date,
  
  -- Workload indicators
  CASE 
    WHEN vs.total_visits > 100 THEN 'High Workload'
    WHEN vs.total_visits BETWEEN 50 AND 100 THEN 'Medium Workload'
    WHEN vs.total_visits < 50 THEN 'Low Workload'
    ELSE 'No Visits'
  END AS workload_category,
  
  -- Performance indicators
  CASE 
    WHEN fs.avg_patient_rating >= 4 THEN 'High Performance'
    WHEN fs.avg_patient_rating BETWEEN 3 AND 4 THEN 'Average Performance'
    WHEN fs.avg_patient_rating < 3 THEN 'Needs Improvement'
    ELSE 'No Rating'
  END AS performance_category,
  
  CURRENT_TIMESTAMP() AS created_at
  
FROM doctor_base db
LEFT JOIN visit_stats vs ON db.doctor_id = vs.doctor_id
LEFT JOIN feedback_stats fs ON db.doctor_id = fs.doctor_id
LEFT JOIN schedule_stats ss ON db.doctor_id = ss.doctor_id
LEFT JOIN hospital_dept_info hd ON db.doctor_id = hd.doctor_id

In [0]:
%sql
-- Display sample records from doctor performance gold table
SELECT 
  doctor_name,
  specialty,
  hospital_name,
  total_visits,
  unique_patients_treated,
  avg_patient_rating,
  total_revenue,
  workload_category,
  performance_category
FROM medical_insurance.gold.doctor_performance
ORDER BY total_visits DESC
LIMIT 10

In [0]:
%sql
-- Summary statistics by specialty
SELECT 
  specialty,
  COUNT(DISTINCT doctor_id) AS total_doctors,
  ROUND(AVG(total_visits), 2) AS avg_visits_per_doctor,
  ROUND(AVG(unique_patients_treated), 2) AS avg_patients_per_doctor,
  ROUND(AVG(avg_patient_rating), 2) AS avg_rating,
  ROUND(AVG(total_revenue), 2) AS avg_revenue_per_doctor,
  ROUND(AVG(avg_waiting_time_minutes), 2) AS avg_wait_time,
  workload_category,
  performance_category
FROM medical_insurance.gold.doctor_performance
GROUP BY specialty, workload_category, performance_category
ORDER BY specialty, workload_category

In [0]:
%sql
select * from medical_insurance.gold.doctor_performance limit 5